In [1]:
using Distributions
using Serialization
using CSV
using Distributed
using ProgressMeter
include("gen_crp_softmax.jl")
# Set random seed for reproducibility

Random.seed!(1777)
n_subs=5


5

### Simulate subjects of a wide range of parameters (Main Simulation)

In [3]:
using Distributions
using CSV
using DataFrames
using Random
using ProgressMeter

include("gen_crp_softmax.jl")

# ---- parameters ----
alpha   = 0.2
n_subs  = 1000
rngseed = 1777

γbase_dist = Uniform(-10, 10)
γcoef_dist = Uniform(-3, 3)
beta_dist  = Uniform(0, 5)
eps_dist   = Uniform(0, 0.1)   # ← new

Random.seed!(rngseed)

all_data = DataFrame()
p = Progress(n_subs, desc="Simulating 1000 subjects...")

for sub_num in 1:n_subs
    Random.seed!(rngseed + sub_num)

    γb  = rand(γbase_dist)
    γc  = rand(γcoef_dist)
    β   = rand(beta_dist)
    ϵ   = rand(eps_dist)   # ← sample ϵ per subject

    # call the model (same as your working version)
    simulated_data, max_clus, total_reward = crp([alpha, γb, γc, β])

    # add identifying parameters
    simulated_data[!, :sub_num]     .= sub_num
    simulated_data[!, :alpha]       .= alpha
    simulated_data[!, :gamma_base]  .= γb
    simulated_data[!, :gamma_coef]  .= γc
    simulated_data[!, :beta]        .= β
    simulated_data[!, :epsilon]     .= ϵ    # ← record the lapse rate

    append!(all_data, simulated_data)
    next!(p)
end

CSV.write("simulated_structure_learning_RANDOM_1000.csv", all_data)
println("✅ Finished! Saved --> simulated_structure_learning_RANDOM_1000.csv")
println("Total subjects simulated: ", n_subs)



Simulating 1000 subjects... 100%|████████████████████████| Time: 18:53:38


✅ Finished! Saved --> simulated_structure_learning_RANDOM_1000.csv
Total subjects simulated: 1000


In [144]:
# Add switch column based on galaxy transitions
function add_switch_column(data)
    switch_col = []
    for i in 1:nrow(data)
        planet = data.planet_num[i]
        curr_planet_type = data.galaxy[i]
        
        if planet > 0
            last_planet_type = data[data.planet_num .== (planet-1), :galaxy][1]
            if curr_planet_type == last_planet_type
                push!(switch_col, String("no_switch"))
            else
                push!(switch_col, String("switch"))
            end
        else
            push!(switch_col, String("no_switch"))
        end
    end
    return switch_col
end

add_switch_column (generic function with 1 method)

In [145]:
alpha = 0.2
gamma_base = 1
gamma_coef = 0
beta = 5
# Create array to store results from each worker
p = Progress(n_subs, desc="Simulating subjects...")
results = @distributed (vcat) for sub_num in 1:n_subs
    simulated_data, max_clus, total_reward = crp([alpha, gamma_base, gamma_coef, beta])
    switch_col = add_switch_column(simulated_data)
    insertcols!(simulated_data, :switch => switch_col)
    simulated_data[!, :sub_num] .= sub_num
    next!(p)
    simulated_data
end
# Save simulated data to CSV
CSV.write("results/model_predictions/simulated_data_uncertainty_non_adapter.csv", results)


Simulating subjects... 100%|█████████████████████████████| Time: 0:10:01


"results/model_predictions/simulated_data_uncertainty_non_adapter.csv"

In [148]:
alpha = 0.2
gamma_base = 1
gamma_coef = -0.3
beta = 5
# Create array to store results from each worker
p = Progress(n_subs, desc="Simulating subjects...")
results = @distributed (vcat) for sub_num in 1:n_subs
    simulated_data, max_clus, total_reward = crp([alpha, gamma_base, gamma_coef, beta])
    switch_col = add_switch_column(simulated_data)
    insertcols!(simulated_data, :switch => switch_col)
    simulated_data[!, :sub_num] .= sub_num
    next!(p)
    simulated_data
end
# Save simulated data to CSV
CSV.write("results/model_predictions/simulated_data_uncertainty_adapter.csv", results)


Simulating subjects... 100%|█████████████████████████████| Time: 0:09:29


"results/model_predictions/simulated_data_uncertainty_adapter.csv"